Twiddles:  [ 1.00000000e+00+0.j          9.23879533e-01-0.38268343j
  7.07106781e-01-0.70710678j  3.82683432e-01-0.92387953j
  6.12323400e-17-1.j         -3.82683432e-01-0.92387953j
 -7.07106781e-01-0.70710678j -9.23879533e-01-0.38268343j]
Twiddle Repeated:  [ 1.00000000e+00+0.j          9.23879533e-01-0.38268343j
  7.07106781e-01-0.70710678j  3.82683432e-01-0.92387953j
  6.12323400e-17-1.j         -3.82683432e-01-0.92387953j
 -7.07106781e-01-0.70710678j -9.23879533e-01-0.38268343j]

Twiddles:  [ 1.00000000e+00+0.j          7.07106781e-01-0.70710678j
  6.12323400e-17-1.j         -7.07106781e-01-0.70710678j]
Twiddle Repeated:  [ 1.00000000e+00+0.j          7.07106781e-01-0.70710678j
  6.12323400e-17-1.j         -7.07106781e-01-0.70710678j
  1.00000000e+00+0.j          7.07106781e-01-0.70710678j
  6.12323400e-17-1.j         -7.07106781e-01-0.70710678j]

Twiddles:  [1.000000e+00+0.j 6.123234e-17-1.j]
Twiddle Repeated:  [1.000000e+00+0.j 6.123234e-17-1.j 1.000000e+00+0.j 6.123234e-17-1.j
 

STD MDC 

In [3]:
import numpy as np

def bit_reverse(x):
    """
    Rearranges the sequence in bit-reversed order.
    DIF FFTs natively output data in bit-reversed frequency bins.
    """
    N = len(x)
    num_bits = int(np.log2(N))
    result = np.zeros_like(x, dtype=complex)
    
    for i in range(N):
        # Reverse the binary representation of the index
        rev_i = int('{:0{width}b}'.format(i, width=num_bits)[::-1], 2)
        result[rev_i] = x[i]
        
    return result

def radix2_dif_mdc_fft(x):
    """
    Simulates a Multi-path Delay Commutator (MDC) Radix-2 DIF FFT.
    """
    x = np.asarray(x, dtype=complex)
    N = len(x)
    stages = int(np.log2(N))
    
    # 1. PARALLEL DATA STREAMS
    # MDC takes in two parallel streams. We simulate this by splitting 
    # the initial input into the top path (first half) and bottom path (second half).
    stream_top = x[:N//2].copy()
    stream_bot = x[N//2:].copy()
    
    # Flow through the hardware pipeline stages
    for stage in range(stages):
        L = N >> stage          # Effective FFT size for this stage
        half_L = L // 2         # Distance between butterfly inputs
        
        # 2. BUTTERFLY UNIT
        # Processes the aligned top and bottom streams in parallel
        bf_out_top = stream_top + stream_bot
        bf_out_bot = stream_top - stream_bot
        
        # 3. MULTIPLIER (Twiddle Factors)
        # As seen in the MDC diagram, the multiplier is ONLY on the bottom branch.
        # We generate the twiddles for this stage and repeat them across the stream.
        twiddles = np.exp(-2j * np.pi * np.arange(half_L) / L)
        twiddles_repeated = np.tile(twiddles, len(bf_out_bot) // half_L)

        # print("Twiddles: ",twiddles)
        # print("Twiddle Repeated: ",twiddles_repeated)
        # print()
        
        bf_out_bot = bf_out_bot * twiddles_repeated
        
        # 4. MULTI-PATH DELAY COMMUTATOR (Data Routing)
        # The commutator switches the data streams to correctly align pairs 
        # for the butterfly unit in the NEXT stage.
        if stage < stages - 1:
            next_half_L = half_L // 2
            new_top = np.zeros_like(stream_top)
            new_bot = np.zeros_like(stream_bot)
            
            # The switch toggles every 'next_half_L' elements
            for i in range(0, len(stream_top), half_L):
                # Pass-through mode (parallel routing)
                new_top[i : i + next_half_L] = bf_out_top[i : i + next_half_L]
                new_bot[i + next_half_L : i + half_L] = bf_out_bot[i + next_half_L : i + half_L]
                
                # Cross-over mode (diagonal routing)
                new_top[i + next_half_L : i + half_L] = bf_out_bot[i : i + next_half_L]
                new_bot[i : i + next_half_L] = bf_out_top[i + next_half_L : i + half_L]
                
            # Advance to the next hardware stage
            stream_top = new_top
            stream_bot = new_bot
        else:
            # No commutator needed on the very last stage
            stream_top = bf_out_top
            stream_bot = bf_out_bot

    # Combine the parallel hardware streams back into a single array
    out_natural = np.empty_like(x)
    # The outputs are interleaved from the top and bottom paths
    out_natural[0::2] = stream_top
    out_natural[1::2] = stream_bot
    
    # 5. BIT REVERSAL
    return bit_reverse(out_natural)

# ==========================================
# Test Bench: Compare with NumPy's FFT
# ==========================================
if __name__ == "__main__":
    # Create a random 16-point input signal
    N = 16
    test_signal = np.random.rand(N) + 1j * np.random.rand(N)
    
    # Run our MDC architecture simulation
    custom_fft_result = radix2_dif_mdc_fft(test_signal)
    
    # Run the standard NumPy FFT
    numpy_fft_result = np.fft.fft(test_signal)
    
    # Check if they match
    difference = np.max(np.abs(custom_fft_result - numpy_fft_result))
    
    print(f"Maximum difference between NumPy and custom MDC FFT: {difference:.2e}")
    if difference < 1e-10:
        print("Success! The MDC hardware simulation perfectly matches the theoretical FFT.")
    else:
        print("Mismatch detected.")

Maximum difference between NumPy and custom MDC FFT: 4.44e-16
Success! The MDC hardware simulation perfectly matches the theoretical FFT.


STD MDC (INT8)

In [4]:
import numpy as np

def bit_reverse_split(x_r, x_i):
    """
    Rearranges the separated real and imaginary sequences in bit-reversed order.
    """
    N = len(x_r)
    num_bits = int(np.log2(N))
    
    res_r = np.zeros_like(x_r, dtype=np.int8)
    res_i = np.zeros_like(x_i, dtype=np.int8)
    
    for i in range(N):
        rev_i = int('{:0{width}b}'.format(i, width=num_bits)[::-1], 2)
        res_r[rev_i] = x_r[i]
        res_i[rev_i] = x_i[i]
        
    return res_r, res_i

def radix2_dif_mdc_fft_int8(x_real, x_imag):
    """
    Simulates a Fixed-Point Int8 MDC Radix-2 DIF FFT.
    """
    N = len(x_real)
    stages = int(np.log2(N))
    
    # 1. PARALLEL DATA STREAMS (Split into Real and Imaginary)
    stream_top_r = x_real[:N//2].copy()
    stream_top_i = x_imag[:N//2].copy()
    stream_bot_r = x_real[N//2:].copy()
    stream_bot_i = x_imag[N//2:].copy()
    
    for stage in range(stages):
        L = N >> stage          
        half_L = L // 2         
        
        # 2. BUTTERFLY UNIT
        # We must upcast to int16 temporarily to prevent overflow during addition.
        # Then we divide by 2 (shift right by 1) to scale it back down to fit in int8.
        top_r_16 = stream_top_r.astype(np.int16)
        top_i_16 = stream_top_i.astype(np.int16)
        bot_r_16 = stream_bot_r.astype(np.int16)
        bot_i_16 = stream_bot_i.astype(np.int16)
        
        bf_out_top_r = ((top_r_16 + bot_r_16) >> 1).astype(np.int8)
        bf_out_top_i = ((top_i_16 + bot_i_16) >> 1).astype(np.int8)
        
        bf_out_bot_r = ((top_r_16 - bot_r_16) >> 1).astype(np.int8)
        bf_out_bot_i = ((top_i_16 - bot_i_16) >> 1).astype(np.int8)
        
        # 3. MULTIPLIER (Fixed-Point Twiddle Factors)
        # Generate floating point angles, but convert them to Q1.7 fixed-point integers
        angles = -2 * np.pi * np.arange(half_L) / L
        twiddles_r_float = np.cos(angles)
        twiddles_i_float = np.sin(angles)
        
        # Scale by 127 and convert to int16 (to prevent overflow during multiplication)
        tw_r_int = np.round(twiddles_r_float * 127).astype(np.int16)
        tw_i_int = np.round(twiddles_i_float * 127).astype(np.int16)
        
        tw_r_rep = np.tile(tw_r_int, len(bf_out_bot_r) // half_L)
        tw_i_rep = np.tile(tw_i_int, len(bf_out_bot_i) // half_L)
        
        # Complex Multiplication: (R1 + jI1) * (R2 + jI2) = (R1R2 - I1I2) + j(R1I2 + I1R2)
        # An 8-bit * 8-bit multiplication yields a 16-bit number. 
        temp_bot_r = bf_out_bot_r.astype(np.int16)
        temp_bot_i = bf_out_bot_i.astype(np.int16)
        
        mult_r = (temp_bot_r * tw_r_rep - temp_bot_i * tw_i_rep)
        mult_i = (temp_bot_r * tw_i_rep + temp_bot_i * tw_r_rep)
        
        # Shift right by 7 to remove the Q1.7 scaling factor we added to the twiddles
        bf_out_bot_r = (mult_r >> 7).astype(np.int8)
        bf_out_bot_i = (mult_i >> 7).astype(np.int8)
        
        # 4. MULTI-PATH DELAY COMMUTATOR (Data Routing)
        if stage < stages - 1:
            next_half_L = half_L // 2
            
            new_top_r = np.zeros_like(stream_top_r)
            new_top_i = np.zeros_like(stream_top_i)
            new_bot_r = np.zeros_like(stream_bot_r)
            new_bot_i = np.zeros_like(stream_bot_i)
            
            for i in range(0, len(stream_top_r), half_L):
                # Pass-through mode
                new_top_r[i : i + next_half_L] = bf_out_top_r[i : i + next_half_L]
                new_top_i[i : i + next_half_L] = bf_out_top_i[i : i + next_half_L]
                new_bot_r[i + next_half_L : i + half_L] = bf_out_bot_r[i + next_half_L : i + half_L]
                new_bot_i[i + next_half_L : i + half_L] = bf_out_bot_i[i + next_half_L : i + half_L]
                
                # Cross-over mode
                new_top_r[i + next_half_L : i + half_L] = bf_out_bot_r[i : i + next_half_L]
                new_top_i[i + next_half_L : i + half_L] = bf_out_bot_i[i : i + next_half_L]
                new_bot_r[i : i + next_half_L] = bf_out_top_r[i + next_half_L : i + half_L]
                new_bot_i[i : i + next_half_L] = bf_out_top_i[i + next_half_L : i + half_L]
                
            stream_top_r, stream_top_i = new_top_r, new_top_i
            stream_bot_r, stream_bot_i = new_bot_r, new_bot_i
        else:
            stream_top_r, stream_top_i = bf_out_top_r, bf_out_top_i
            stream_bot_r, stream_bot_i = bf_out_bot_r, bf_out_bot_i

    # Combine streams
    out_nat_r = np.empty_like(x_real)
    out_nat_i = np.empty_like(x_imag)
    
    out_nat_r[0::2], out_nat_r[1::2] = stream_top_r, stream_bot_r
    out_nat_i[0::2], out_nat_i[1::2] = stream_top_i, stream_bot_i
    
    # 5. BIT REVERSAL
    return bit_reverse_split(out_nat_r, out_nat_i)

# ==========================================
# Test Bench: Compare Int8 hardware vs Float
# ==========================================
if __name__ == "__main__":
    N = 16
    
    # Create a random input signal constrained to int8 safe bounds (-128 to 127)
    x_real_in = np.random.randint(-128, 127, N, dtype=np.int8)
    x_imag_in = np.random.randint(-128, 127, N, dtype=np.int8)
    
    # Run the custom INT8 FFT
    res_r, res_i = radix2_dif_mdc_fft_int8(x_real_in, x_imag_in)
    
    # Run NumPy FFT for comparison
    # Note: We must divide the numpy input by 2^stages to match the hardware bit-shifting!
    float_input = x_real_in + 1j * x_imag_in
    numpy_fft_result = np.fft.fft(float_input) / (2 ** int(np.log2(N)))
    
    print("Int8 Hardware Simulation Results (First 4 bins):")
    for i in range(4):
        print(f"Bin {i}: {res_r[i]} + {res_i[i]}j")
        
    print("\nIdeal Floating Point Results (Scaled):")
    for i in range(4):
        print(f"Bin {i}: {numpy_fft_result[i].real:.1f} + {numpy_fft_result[i].imag:.1f}j")

    # Calculate average error
    hardware_complex = res_r + 1j * res_i
    error = np.abs(hardware_complex - numpy_fft_result)
    print(f"\nAverage Quantization Error per bin: {np.mean(error):.2f}")

Int8 Hardware Simulation Results (First 4 bins):
Bin 0: 46 + -7j
Bin 1: -7 + 2j
Bin 2: 26 + -4j
Bin 3: -17 + -3j

Ideal Floating Point Results (Scaled):
Bin 0: 46.7 + -5.5j
Bin 1: -5.7 + 2.8j
Bin 2: 27.8 + -3.5j
Bin 3: -16.0 + -1.6j

Average Quantization Error per bin: 1.67


STD MDC (INT16)

In [1]:
import numpy as np

def bit_reverse_split(x_r, x_i):
    """
    Rearranges the separated real and imaginary sequences in bit-reversed order.
    """
    N = len(x_r)
    num_bits = int(np.log2(N))
    
    res_r = np.zeros_like(x_r, dtype=np.int16)
    res_i = np.zeros_like(x_i, dtype=np.int16)
    
    for i in range(N):
        rev_i = int('{:0{width}b}'.format(i, width=num_bits)[::-1], 2)
        res_r[rev_i] = x_r[i]
        res_i[rev_i] = x_i[i]
        
    return res_r, res_i

def radix2_dif_mdc_fft_int16(x_real, x_imag):
    """
    Simulates a Fixed-Point Int16 MDC Radix-2 DIF FFT.
    """
    N = len(x_real)
    stages = int(np.log2(N))
    
    # 1. PARALLEL DATA STREAMS (Split into Real and Imaginary)
    stream_top_r = x_real[:N//2].copy()
    stream_top_i = x_imag[:N//2].copy()
    stream_bot_r = x_real[N//2:].copy()
    stream_bot_i = x_imag[N//2:].copy()
    
    for stage in range(stages):
        L = N >> stage          
        half_L = L // 2         
        
        # 2. BUTTERFLY UNIT
        # We must upcast to int32 temporarily to prevent overflow during addition.
        # Then we divide by 2 (shift right by 1) to scale it back down to fit in int16.
        top_r_32 = stream_top_r.astype(np.int32)
        top_i_32 = stream_top_i.astype(np.int32)
        bot_r_32 = stream_bot_r.astype(np.int32)
        bot_i_32 = stream_bot_i.astype(np.int32)
        
        bf_out_top_r = ((top_r_32 + bot_r_32) >> 1).astype(np.int16)
        bf_out_top_i = ((top_i_32 + bot_i_32) >> 1).astype(np.int16)
        
        bf_out_bot_r = ((top_r_32 - bot_r_32) >> 1).astype(np.int16)
        bf_out_bot_i = ((top_i_32 - bot_i_32) >> 1).astype(np.int16)
        
        # 3. MULTIPLIER (Fixed-Point Twiddle Factors)
        # Generate floating point angles, but convert them to Q1.15 fixed-point integers
        angles = -2 * np.pi * np.arange(half_L) / L
        twiddles_r_float = np.cos(angles)
        twiddles_i_float = np.sin(angles)
        
        # Scale by 32767 and convert to int16 (to map to full 16-bit signed range)
        tw_r_int = np.round(twiddles_r_float * 32767).astype(np.int16)
        tw_i_int = np.round(twiddles_i_float * 32767).astype(np.int16)
        
        tw_r_rep = np.tile(tw_r_int, len(bf_out_bot_r) // half_L)
        tw_i_rep = np.tile(tw_i_int, len(bf_out_bot_i) // half_L)
        
        # Complex Multiplication: (R1 + jI1) * (R2 + jI2) = (R1R2 - I1I2) + j(R1I2 + I1R2)
        # A 16-bit * 16-bit multiplication yields a 32-bit number. 
        temp_bot_r = bf_out_bot_r.astype(np.int32)
        temp_bot_i = bf_out_bot_i.astype(np.int32)
        tw_r_rep_32 = tw_r_rep.astype(np.int32)
        tw_i_rep_32 = tw_i_rep.astype(np.int32)
        
        mult_r = (temp_bot_r * tw_r_rep_32 - temp_bot_i * tw_i_rep_32)
        mult_i = (temp_bot_r * tw_i_rep_32 + temp_bot_i * tw_r_rep_32)
        
        # Shift right by 15 to remove the Q1.15 scaling factor we added to the twiddles
        bf_out_bot_r = (mult_r >> 15).astype(np.int16)
        bf_out_bot_i = (mult_i >> 15).astype(np.int16)
        
        # 4. MULTI-PATH DELAY COMMUTATOR (Data Routing)
        if stage < stages - 1:
            next_half_L = half_L // 2
            
            new_top_r = np.zeros_like(stream_top_r)
            new_top_i = np.zeros_like(stream_top_i)
            new_bot_r = np.zeros_like(stream_bot_r)
            new_bot_i = np.zeros_like(stream_bot_i)
            
            for i in range(0, len(stream_top_r), half_L):
                # Pass-through mode
                new_top_r[i : i + next_half_L] = bf_out_top_r[i : i + next_half_L]
                new_top_i[i : i + next_half_L] = bf_out_top_i[i : i + next_half_L]
                new_bot_r[i + next_half_L : i + half_L] = bf_out_bot_r[i + next_half_L : i + half_L]
                new_bot_i[i + next_half_L : i + half_L] = bf_out_bot_i[i + next_half_L : i + half_L]
                
                # Cross-over mode
                new_top_r[i + next_half_L : i + half_L] = bf_out_bot_r[i : i + next_half_L]
                new_top_i[i + next_half_L : i + half_L] = bf_out_bot_i[i : i + next_half_L]
                new_bot_r[i : i + next_half_L] = bf_out_top_r[i + next_half_L : i + half_L]
                new_bot_i[i : i + next_half_L] = bf_out_top_i[i + next_half_L : i + half_L]
                
            stream_top_r, stream_top_i = new_top_r, new_top_i
            stream_bot_r, stream_bot_i = new_bot_r, new_bot_i
        else:
            stream_top_r, stream_top_i = bf_out_top_r, bf_out_top_i
            stream_bot_r, stream_bot_i = bf_out_bot_r, bf_out_bot_i

    # Combine streams
    out_nat_r = np.empty_like(x_real)
    out_nat_i = np.empty_like(x_imag)
    
    out_nat_r[0::2], out_nat_r[1::2] = stream_top_r, stream_bot_r
    out_nat_i[0::2], out_nat_i[1::2] = stream_top_i, stream_bot_i
    
    # 5. BIT REVERSAL
    return bit_reverse_split(out_nat_r, out_nat_i)

# ==========================================
# Test Bench: Compare Int16 hardware vs Float
# ==========================================

if __name__ == "__main__":
    N = 16
    
    # Restrict generation to half-scale to prevent complex rotation overflow
    # Max magnitude is now sqrt(16383^2 + 16383^2) = ~23169 (safely < 32767)
    x_real_in = np.random.randint(-16384, 16383, N, dtype=np.int16)
    x_imag_in = np.random.randint(-16384, 16383, N, dtype=np.int16)
    
    # Run the custom INT16 FFT
    res_r, res_i = radix2_dif_mdc_fft_int16(x_real_in, x_imag_in)
    
    # Run NumPy FFT for comparison
    float_input = x_real_in + 1j * x_imag_in
    numpy_fft_result = np.fft.fft(float_input) / (2 ** int(np.log2(N)))
    
    print("Int16 Hardware Simulation Results (First 4 bins):")
    for i in range(4):
        print(f"Bin {i}: {res_r[i]} + {res_i[i]}j")
        
    print("\nIdeal Floating Point Results (Scaled):")
    for i in range(4):
        print(f"Bin {i}: {numpy_fft_result[i].real:.1f} + {numpy_fft_result[i].imag:.1f}j")

    # Calculate average error
    hardware_complex = res_r + 1j * res_i
    error = np.abs(hardware_complex - numpy_fft_result)
    print(f"\nAverage Quantization Error per bin: {np.mean(error):.2f}")

Int16 Hardware Simulation Results (First 4 bins):
Bin 0: 3053 + 3002j
Bin 1: 1190 + 1270j
Bin 2: 644 + 521j
Bin 3: 2460 + -1342j

Ideal Floating Point Results (Scaled):
Bin 0: 3054.3 + 3002.9j
Bin 1: 1191.5 + 1271.5j
Bin 2: 645.7 + 522.2j
Bin 3: 2461.3 + -1341.7j

Average Quantization Error per bin: 1.46


PROPOSED R2MDC

In [5]:
import numpy as np

def bit_reverse(x):
    """Rearranges the final 64-point sequence in bit-reversed order."""
    N = len(x)
    num_bits = int(np.log2(N))
    result = np.zeros_like(x, dtype=complex)
    for i in range(N):
        rev_i = int('{:0{width}b}'.format(i, width=num_bits)[::-1], 2)
        result[rev_i] = x[i]
    return result

def mdc_commutator_routing(stream_top, stream_bot, half_L):
    """
    Simulates the physical data routing of an MDC switch and delay lines.
    Instead of simulating cycle-by-cycle 'junk' data, this array slicing 
    fast-forwards the time-alignment to pair the correct valid elements.
    """
    next_half_L = half_L // 2
    new_top = np.zeros_like(stream_top)
    new_bot = np.zeros_like(stream_bot)
    
    for i in range(0, len(stream_top), half_L):
        # Switch in Pass-through mode
        new_top[i : i + next_half_L] = stream_top[i : i + next_half_L]
        new_bot[i + next_half_L : i + half_L] = stream_bot[i + next_half_L : i + half_L]
        
        # Switch in Cross-over mode
        new_top[i + next_half_L : i + half_L] = stream_bot[i : i + next_half_L]
        new_bot[i : i + next_half_L] = stream_top[i + next_half_L : i + half_L]
        
    return new_top, new_bot

def proposed_8parallel_r2mdc_fft(x):
    """
    Simulates the 64-point 8-parallel FFT architecture.
    """
    x = np.asarray(x, dtype=complex)
    
    # Initialize 4 parallel datapaths (8 hardware streams total)
    # This aligns with the input distribution described in Fig. 5.
    dp1_t = x[0:8].copy();  dp1_b = x[32:40].copy()
    dp2_t = x[8:16].copy(); dp2_b = x[40:48].copy()
    dp3_t = x[16:24].copy();dp3_b = x[48:56].copy()
    dp4_t = x[24:32].copy();dp4_b = x[56:64].copy()
    
    # ==========================================
    # STAGE 1 (Gap = 32)
    # ==========================================
    tw1 = np.exp(-2j * np.pi * np.arange(32) / 64)
    
    # Parallel Butterflies
    bf1_dp1_t = dp1_t + dp1_b; bf1_dp1_b = (dp1_t - dp1_b) * tw1[0:8]
    bf1_dp2_t = dp2_t + dp2_b; bf1_dp2_b = (dp2_t - dp2_b) * tw1[8:16]
    bf1_dp3_t = dp3_t + dp3_b; bf1_dp3_b = (dp3_t - dp3_b) * tw1[16:24]
    bf1_dp4_t = dp4_t + dp4_b; bf1_dp4_b = (dp4_t - dp4_b) * tw1[24:32]
    
    # Cross-wiring between Stage 1 and Stage 2 (from Fig 5)
    st2_dp1_t = bf1_dp1_t; st2_dp1_b = bf1_dp3_t
    st2_dp2_t = bf1_dp2_t; st2_dp2_b = bf1_dp4_t
    st2_dp3_t = bf1_dp1_b; st2_dp3_b = bf1_dp3_b
    st2_dp4_t = bf1_dp2_b; st2_dp4_b = bf1_dp4_b
    
    # ==========================================
    # STAGE 2 (Gap = 16)
    # ==========================================
    tw2 = np.exp(-2j * np.pi * np.arange(16) / 32)
    
    # Parallel Butterflies. Notice how DP3 reuses DP1's ROM addresses, 
    # and DP4 reuses DP2's ROM addresses as outlined in the paper.
    bf2_dp1_t = st2_dp1_t + st2_dp1_b; bf2_dp1_b = (st2_dp1_t - st2_dp1_b) * tw2[0:8]
    bf2_dp2_t = st2_dp2_t + st2_dp2_b; bf2_dp2_b = (st2_dp2_t - st2_dp2_b) * tw2[8:16]
    bf2_dp3_t = st2_dp3_t + st2_dp3_b; bf2_dp3_b = (st2_dp3_t - st2_dp3_b) * tw2[0:8]
    bf2_dp4_t = st2_dp4_t + st2_dp4_b; bf2_dp4_b = (st2_dp4_t - st2_dp4_b) * tw2[8:16]
    
    # Cross-wiring between Stage 2 and Stage 3. 
    # "The outputs from the lower ports of the butterflies of datapaths I and III 
    # are fed to datapath II and IV... outputs from the upper ports of II and IV 
    # are fed to datapath I and III"
    st3_dp1_t = bf2_dp1_t; st3_dp1_b = bf2_dp2_t
    st3_dp2_t = bf2_dp1_b; st3_dp2_b = bf2_dp2_b
    st3_dp3_t = bf2_dp3_t; st3_dp3_b = bf2_dp4_t
    st3_dp4_t = bf2_dp3_b; st3_dp4_b = bf2_dp4_b
    
    # ==========================================
    # STAGES 3, 4, 5, 6
    # ==========================================
    # After Stage 2 routing, each datapath has successfully isolated an 
    # independent 16-point FFT sequence. 
    tw3 = np.exp(-2j * np.pi * np.arange(8) / 16)
    
    def process_mdc_sub_fft(t_in, b_in):
        """Processes the remaining stages using standard MDC logic."""
        # Stage 3 (Gap = 8): Processed directly without routing delays
        t3 = t_in + b_in; b3 = (t_in - b_in) * tw3
        
        # Stage 4 (Gap = 4)
        t4_r, b4_r = mdc_commutator_routing(t3, b3, half_L=8)
        tw4 = np.tile(np.exp(-2j * np.pi * np.arange(4) / 8), 2)
        t4 = t4_r + b4_r; b4 = (t4_r - b4_r) * tw4
        
        # Stage 5 (Gap = 2)
        t5_r, b5_r = mdc_commutator_routing(t4, b4, half_L=4)
        tw5 = np.tile(np.exp(-2j * np.pi * np.arange(2) / 4), 4)
        t5 = t5_r + b5_r; b5 = (t5_r - b5_r) * tw5
        
        # Stage 6 (Gap = 1)
        t6_r, b6_r = mdc_commutator_routing(t5, b5, half_L=2)
        tw6 = np.ones(8)
        t6 = t6_r + b6_r; b6 = (t6_r - b6_r) * tw6
        
        # Merge the parallel data streams
        out = np.zeros(16, dtype=complex)
        out[0::2] = t6
        out[1::2] = b6
        return out
        
    out_dp1 = process_mdc_sub_fft(st3_dp1_t, st3_dp1_b)
    out_dp2 = process_mdc_sub_fft(st3_dp2_t, st3_dp2_b)
    out_dp3 = process_mdc_sub_fft(st3_dp3_t, st3_dp3_b)
    out_dp4 = process_mdc_sub_fft(st3_dp4_t, st3_dp4_b)
    
    # Combine the 4 independent sub-FFT streams
    final_out = np.concatenate((out_dp1, out_dp2, out_dp3, out_dp4))
    
    return bit_reverse(final_out)

# ==========================================
# Test Bench
# ==========================================
if __name__ == "__main__":
    test_signal = np.random.rand(64) + 1j * np.random.rand(64)
    
    custom_fft_result = proposed_8parallel_r2mdc_fft(test_signal)
    numpy_fft_result = np.fft.fft(test_signal)
    
    diff = np.max(np.abs(custom_fft_result - numpy_fft_result))
    
    print(f"Maximum quantization error: {diff:.2e}")
    if diff < 1e-10:
        print("Success! The proposed 8-parallel architecture simulation perfectly matches the theoretical FFT.")

Maximum quantization error: 1.74e-15
Success! The proposed 8-parallel architecture simulation perfectly matches the theoretical FFT.


GENERAL PROPOSED R2MDC

In [2]:
import numpy as np

def bit_reverse(x):
    """Rearranges the final sequence in bit-reversed order."""
    N = len(x)
    num_bits = int(np.log2(N))
    result = np.zeros_like(x, dtype=complex)
    for i in range(N):
        rev_i = int('{:0{width}b}'.format(i, width=num_bits)[::-1], 2)
        result[rev_i] = x[i]
    return result

def mdc_commutator_routing(stream_top, stream_bot, half_L):
    """Simulates standard temporal routing via an MDC 2x2 switch and delay lines."""
    next_half_L = half_L // 2
    new_top = np.zeros_like(stream_top)
    new_bot = np.zeros_like(stream_bot)
    
    for i in range(0, len(stream_top), half_L):
        # Pass-through
        new_top[i : i + next_half_L] = stream_top[i : i + next_half_L]
        new_bot[i + next_half_L : i + half_L] = stream_bot[i + next_half_L : i + half_L]
        # Cross-over
        new_top[i + next_half_L : i + half_L] = stream_bot[i : i + next_half_L]
        new_bot[i : i + next_half_L] = stream_top[i + next_half_L : i + half_L]
        
    return new_top, new_bot

def generalized_parallel_mdc_fft(x, P):
    """
    Simulates a hardware accelerator for an N-point FFT processing P parallel samples per cycle.
    """
    x = np.asarray(x, dtype=complex)
    N = len(x)
    
    assert N & (N - 1) == 0, "N must be a power of 2"
    assert P & (P - 1) == 0, "P must be a power of 2"
    assert P >= 2, "Parallelism P must be at least 2"
    assert P <= N, "Parallelism P cannot exceed FFT size N"
    
    D = P // 2                 # Number of physical Datapaths
    chunk_size = N // P        # Number of clock cycles needed to load data
    spatial_stages = int(np.log2(D))
    
    # ==========================================
    # STEP 1: INITIAL DATA DISTRIBUTION (Input Offsets)
    # ==========================================
    DP = []
    for d in range(D):
        t_start = d * chunk_size
        b_start = t_start + (N // 2)
        DP.append({
            't': x[t_start : t_start + chunk_size].copy(),
            'b': x[b_start : b_start + chunk_size].copy()
        })
        
    # ==========================================
    # STEP 2: SPATIAL ROUTING PHASE (Stages 1 to log2(D))
    # ==========================================
    for stage in range(spatial_stages):
        L = N >> stage          # Effective FFT size for this stage
        half_L = L // 2         # Distance between butterfly inputs
        
        # Calculate full twiddle array for this stage
        tw = np.exp(-2j * np.pi * np.arange(half_L) / L)
        
        new_DP = [{'t': None, 'b': None} for _ in range(D)]
        bf_out = []
        
        # A. Execute Parallel Butterflies
        for d in range(D):
            t_in, b_in = DP[d]['t'], DP[d]['b']
            
            # Hardware dynamically shares/reuses ROM addresses based on datapath index
            tw_start = (d * chunk_size) % half_L
            tw_slice = tw[tw_start : tw_start + chunk_size]
            
            t_out = t_in + b_in
            b_out = (t_in - b_in) * tw_slice
            bf_out.append({'t': t_out, 'b': b_out})
            
        # B. Execute Hardwired Cross-Routing
        delta_d = D >> (stage + 1)  # The swap distance: D/(2^s)
        
        for d in range(D):
            # Check if this datapath is the 'Upper' path of a swap pair
            if (d // delta_d) % 2 == 0:
                d_partner = d + delta_d
                
                # Perform the physical wire crossing
                new_DP[d]['t']         = bf_out[d]['t']
                new_DP[d]['b']         = bf_out[d_partner]['t']
                new_DP[d_partner]['t'] = bf_out[d]['b']
                new_DP[d_partner]['b'] = bf_out[d_partner]['b']
                
        DP = new_DP

    # ==========================================
    # STEP 3: TEMPORAL ROUTING PHASE (Standard MDC)
    # ==========================================
    starting_gap = N // (2 * D)
    
    def process_mdc_sub_fft(t_in, b_in):
        """Processes the localized data via standard shift-registers & commutators."""
        current_gap = starting_gap
        t_curr, b_curr = t_in, b_in
        
        while current_gap >= 1:
            # Commutator routing is skipped on the first iteration because the 
            # spatial routing phase already aligned the data for us!
            if current_gap < starting_gap:
                t_curr, b_curr = mdc_commutator_routing(t_curr, b_curr, current_gap * 2)
                
            L_sub = current_gap * 2
            tw_sub = np.exp(-2j * np.pi * np.arange(current_gap) / L_sub)
            tw_repeated = np.tile(tw_sub, len(b_curr) // current_gap)
            
            t_next = t_curr + b_curr
            b_next = (t_curr - b_curr) * tw_repeated
            
            t_curr, b_curr = t_next, b_next
            current_gap //= 2
            
        # Interleave the parallel outputs of this datapath
        out = np.zeros(len(t_curr) * 2, dtype=complex)
        out[0::2] = t_curr
        out[1::2] = b_curr
        return out

    # Execute the temporal phase independently for all D datapaths
    final_streams = []
    for d in range(D):
        final_streams.append(process_mdc_sub_fft(DP[d]['t'], DP[d]['b']))
        
    # Combine the independent streams back into the full sequence
    final_out = np.concatenate(final_streams)
    
    return bit_reverse(final_out)

# ==========================================
# TEST BENCH: Validate generalization
# ==========================================
if __name__ == "__main__":
    configs = [
        {"N": 16, "P": 4},
        {"N": 64, "P": 8},   # The paper's specific architecture
        {"N": 256, "P": 16}, # A highly parallel scale-up
        {"N": 1024, "P": 2}  # A standard dual-wire setup
    ]
    
    for cfg in configs:
        N, P = cfg["N"], cfg["P"]
        print(f"Testing FFT Size N={N}, Parallelism P={P} (Datapaths={P//2})")
        
        test_signal = np.random.rand(N) + 1j * np.random.rand(N)
        
        custom_result = generalized_parallel_mdc_fft(test_signal, P)
        numpy_result = np.fft.fft(test_signal)
        
        diff = np.max(np.abs(custom_result - numpy_result))
        
        if diff < 1e-10:
            print(f"  -> SUCCESS! Max error: {diff:.2e}\n")
        else:
            print(f"  -> FAILURE! Max error: {diff:.2e}\n")

Testing FFT Size N=16, Parallelism P=4 (Datapaths=2)
  -> SUCCESS! Max error: 2.22e-16

Testing FFT Size N=64, Parallelism P=8 (Datapaths=4)
  -> SUCCESS! Max error: 2.26e-15

Testing FFT Size N=256, Parallelism P=16 (Datapaths=8)
  -> SUCCESS! Max error: 5.69e-15

Testing FFT Size N=1024, Parallelism P=2 (Datapaths=1)
  -> SUCCESS! Max error: 1.90e-14



SIMPLE Radix-2

In [7]:
import math
import cmath
import numpy as np

def reverse_bits(n, num_bits):
    """Simulates the combinational bit-reversal routing."""
    result = 0
    for i in range(num_bits):
        result = (result << 1) | (n & 1)
        n >>= 1
    return result

def simulate_hw_fft(x_in, N=128):
    """Simulates the N-point Radix-2 DIT FFT hardware pipeline."""
    num_bits = int(math.log2(N))
    
    # STAGE 0: Combinational Bit-Reversal
    X = [0j] * N
    for i in range(N):
        rev_i = reverse_bits(i, num_bits)
        X[rev_i] = x_in[i]
        
    pipeline_registers = [X.copy()]
    
    # Precompute Twiddles W_N^k
    W = [cmath.exp(-2j * cmath.pi * k / N) for k in range(N // 2)]
    
    # PIPELINE STAGES 1 to log2(N)
    for stage in range(1, num_bits + 1):
        half_step = 1 << (stage - 1)
        step = 1 << stage
        next_X = X.copy()

        print("--------------------------------------------------------------------")
        print("Stage: ",stage, end=" ")
        print("Step: ",step)
        
        for k in range(0, N, step):
            for j in range(half_step):
                twiddle = W[j * (N // step)]
                top_idx = k + j
                bot_idx = k + j + half_step

                print("Top: ",top_idx,end=" ")
                print("Bottom: ",bot_idx, end=" ")
                print("Twiddle idx: ",j * (N // step))

                A = X[top_idx]
                B = X[bot_idx]
                C = B * twiddle
                
                next_X[top_idx] = A + C
                next_X[bot_idx] = A - C
        
        X = next_X
        pipeline_registers.append(X.copy())
        
    return pipeline_registers

def verify_fft_error(hw_output, original_input):
    """
    Compares the custom hardware simulation against Numpy's golden reference.
    """
    # 1. Generate Golden Reference
    golden_fft = np.fft.fft(original_input)
    
    # 2. Convert HW output to Numpy array for easy math
    hw_array = np.array(hw_output)
    
    # 3. Calculate metrics
    differences = hw_array - golden_fft
    abs_errors = np.abs(differences)
    
    max_error = np.max(abs_errors)
    mse = np.mean(abs_errors**2)
    
    return golden_fft, max_error, mse

# =====================================================================
# Test the Simulation and Verify Error
# =====================================================================
if __name__ == "__main__":
    N = 128 
    print(f"--- Simulating and Verifying {N}-point Hardware FFT ---")
    
    # Create test signal: DC + two sine waves + noise
    np.random.seed(42)
    x = [
        1.0 + math.sin(2 * math.pi * 5 * i / N) + 
        0.5 * math.cos(2 * math.pi * 12 * i / N) + 
        complex(np.random.normal(0, 0.1), np.random.normal(0, 0.1))
        for i in range(N)
    ]
    
    # Run HW Simulation
    states = simulate_hw_fft(x, N)
    final_hw_output = states[-1] # Extract the final pipeline stage
    
    # Calculate Error
    golden_ref, max_err, mse = verify_fft_error(final_hw_output, x)
    
    # print("\n[Verification Results]")
    # print(f"Max Absolute Error: {max_err:.4e}")
    # print(f"Mean Squared Error: {mse:.4e}")
    
    # Optional: Display a few bins to visually confirm
    # print("\n[Sample Bin Comparison (First 4 bins)]")
    # print(f"{'Bin':<5} | {'HW Simulation':<35} | {'Numpy Golden':<35}")
    # print("-" * 80)
    for i in range(4):
        hw_val = final_hw_output[i]
        np_val = golden_ref[i]
        # print(f"{i:<5} | {hw_val.real:>10.4f} + {hw_val.imag:>10.4f}j | {np_val.real:>10.4f} + {np_val.imag:>10.4f}j")

--- Simulating and Verifying 128-point Hardware FFT ---
--------------------------------------------------------------------
Stage:  1 Step:  2
Top:  0 Bottom:  1 Twiddle idx:  0
Top:  2 Bottom:  3 Twiddle idx:  0
Top:  4 Bottom:  5 Twiddle idx:  0
Top:  6 Bottom:  7 Twiddle idx:  0
Top:  8 Bottom:  9 Twiddle idx:  0
Top:  10 Bottom:  11 Twiddle idx:  0
Top:  12 Bottom:  13 Twiddle idx:  0
Top:  14 Bottom:  15 Twiddle idx:  0
Top:  16 Bottom:  17 Twiddle idx:  0
Top:  18 Bottom:  19 Twiddle idx:  0
Top:  20 Bottom:  21 Twiddle idx:  0
Top:  22 Bottom:  23 Twiddle idx:  0
Top:  24 Bottom:  25 Twiddle idx:  0
Top:  26 Bottom:  27 Twiddle idx:  0
Top:  28 Bottom:  29 Twiddle idx:  0
Top:  30 Bottom:  31 Twiddle idx:  0
Top:  32 Bottom:  33 Twiddle idx:  0
Top:  34 Bottom:  35 Twiddle idx:  0
Top:  36 Bottom:  37 Twiddle idx:  0
Top:  38 Bottom:  39 Twiddle idx:  0
Top:  40 Bottom:  41 Twiddle idx:  0
Top:  42 Bottom:  43 Twiddle idx:  0
Top:  44 Bottom:  45 Twiddle idx:  0
Top:  46 Bottom

MIXED RADIX (4,4,4,2) 128-FFT

In [12]:
import cmath
import math
import numpy as np

def mixed_radix_reorder_128(n):
    """
    Stage 0: Hardware Wire Routing
    Reorders a 7-bit index based on Radix-(4,4,4,2) grouping.
    Original bit groups: (b6 b5) (b4 b3) (b2 b1) (b0)
    Reversed bit groups: (b0) (b2 b1) (b4 b3) (b6 b5)
    """
    g3 = (n >> 5) & 0b11  # Top 2 bits (b6, b5)
    g2 = (n >> 3) & 0b11  # Next 2 bits (b4, b3)
    g1 = (n >> 1) & 0b11  # Next 2 bits (b2, b1)
    g0 = n & 0b1          # Bottom 1 bit (b0)
    
    # Recombine in reverse group order
    return (g0 << 6) | (g1 << 4) | (g2 << 2) | g3

def radix4_butterfly(A, B, C, D):
    """
    Calculates the Radix-4 butterfly outputs.
    In hardware, this block uses 8 complex adders and handles 
    multiplications by 'j' purely through wire routing.
    """
    # Intermediate sums (Adder Tree Stage 1)
    E = A + C
    F = A - C
    G = B + D
    H = B - D
    
    # Final outputs (Adder Tree Stage 2)
    X0 = E + G
    X2 = E - G
    X1 = F - 1j * H
    X3 = F + 1j * H
    
    return X0, X1, X2, X3

def simulate_mixed_radix_fft_128(x_in):
    """
    Simulates the N=128 Mixed-Radix Hardware Pipeline.
    """
    N = 128
    
    # Precompute Twiddle Factors: W_N^k = e^(-j * 2*pi * k / N)
    W = [cmath.exp(-2j * cmath.pi * k / N) for k in range(N)]
    
    # STAGE 0: Input Reordering
    X = [0j] * N
    for i in range(N):
        X[mixed_radix_reorder_128(i)] = x_in[i]
        
    # PIPELINE STAGES 1 to 4
    # Format: (Radix, M: previous block size, L: new block size)
    pipeline_stages = [
        (4, 1, 4),      # Stage 1: Radix-4
        (4, 4, 16),     # Stage 2: Radix-4
        (4, 16, 64),    # Stage 3: Radix-4
        (2, 64, 128)    # Stage 4: Radix-2
    ]
    
    for stage_idx, (R, M, L) in enumerate(pipeline_stages):
        next_X = X.copy()
        twiddle_step = N // L
        
        print("------------------------------------------------")
        print("Stage: ",stage_idx,end=" ")
        print("Twiddle Step: ",twiddle_step)

        for k in range(0, N, L):
            for j in range(M):
                if R == 4:
                    # 1. Fetch the 4 inputs for this butterfly
                    idx0, idx1 = k + j, k + j + M
                    idx2, idx3 = k + j + 2*M, k + j + 3*M

                    print("Indices: ",idx0,idx1,idx2,idx3,end="    ")
                    
                    # 2. Apply Twiddle Factors (Multipliers in hardware)
                    A = X[idx0] # W^0 = 1
                    B = X[idx1] * W[1 * j * twiddle_step]
                    C = X[idx2] * W[2 * j * twiddle_step]
                    D = X[idx3] * W[3 * j * twiddle_step]

                    print("W indices: ",1 * j * twiddle_step,2 * j * twiddle_step,3 * j * twiddle_step)
                    
                    # 3. Radix-4 Adder Tree
                    X0, X1, X2, X3 = radix4_butterfly(A, B, C, D)
                    
                    # 4. Route to next pipeline register
                    next_X[idx0] = X0
                    next_X[idx1] = X1
                    next_X[idx2] = X2
                    next_X[idx3] = X3
                    
                elif R == 2:
                    # 1. Fetch the 2 inputs for this butterfly
                    idx0, idx1 = k + j, k + j + M
                    print("Indices: ",idx0,idx1,end="    ")
                    # 2. Apply Twiddle Factor
                    A = X[idx0]
                    B = X[idx1] * W[1 * j * twiddle_step]

                    print("W index: ",1 * j * twiddle_step)
                    
                    # 3. Radix-2 Adder Tree & Route
                    next_X[idx0] = A + B
                    next_X[idx1] = A - B
                    
        # Clock edge: latch the pipeline stage
        X = next_X
        
    return X

# =====================================================================
# Test and Verification
# =====================================================================
if __name__ == "__main__":
    N = 128
    
    # 1. Generate a complex test signal with noise
    np.random.seed(100)
    x = [
        1.0 + math.sin(2 * math.pi * 5 * i / N) + 
        complex(np.random.normal(0, 0.1), np.random.normal(0, 0.1))
        for i in range(N)
    ]
    
    # 2. Run the custom HW simulation
    hw_output = simulate_mixed_radix_fft_128(x)
    
    # 3. Run the Numpy Golden Reference
    golden_output = np.fft.fft(x)
    
    # 4. Calculate error
    differences = np.array(hw_output) - golden_output
    max_error = np.max(np.abs(differences))
    
    print("--- N=128 Mixed-Radix (4,4,4,2) FFT Verification ---")
    print(f"Max Absolute Error vs Numpy: {max_error:.4e}")
    if max_error < 1e-10:
        print("Status: SUCCESS (Error is within floating-point precision bounds)")

------------------------------------------------
Stage:  0 Twiddle Step:  32
Indices:  0 1 2 3    W indices:  0 0 0
Indices:  4 5 6 7    W indices:  0 0 0
Indices:  8 9 10 11    W indices:  0 0 0
Indices:  12 13 14 15    W indices:  0 0 0
Indices:  16 17 18 19    W indices:  0 0 0
Indices:  20 21 22 23    W indices:  0 0 0
Indices:  24 25 26 27    W indices:  0 0 0
Indices:  28 29 30 31    W indices:  0 0 0
Indices:  32 33 34 35    W indices:  0 0 0
Indices:  36 37 38 39    W indices:  0 0 0
Indices:  40 41 42 43    W indices:  0 0 0
Indices:  44 45 46 47    W indices:  0 0 0
Indices:  48 49 50 51    W indices:  0 0 0
Indices:  52 53 54 55    W indices:  0 0 0
Indices:  56 57 58 59    W indices:  0 0 0
Indices:  60 61 62 63    W indices:  0 0 0
Indices:  64 65 66 67    W indices:  0 0 0
Indices:  68 69 70 71    W indices:  0 0 0
Indices:  72 73 74 75    W indices:  0 0 0
Indices:  76 77 78 79    W indices:  0 0 0
Indices:  80 81 82 83    W indices:  0 0 0
Indices:  84 85 86 87    W ind

MIXED RADIX (4/2) 2^N-point FFT

In [25]:
import cmath
import math
import numpy as np

def generic_mixed_radix_reorder(n, num_bits):
    """Stage 0: Dynamic Hardware Wire Routing"""
    temp_n = n
    result = 0
    
    if num_bits % 2 != 0:
        g0 = temp_n & 0b1
        temp_n >>= 1
        current_write_pos = num_bits - 1
        result |= (g0 << current_write_pos)
    else:
        current_write_pos = num_bits
        
    while current_write_pos > 0:
        g = temp_n & 0b11
        temp_n >>= 2
        current_write_pos -= 2
        result |= (g << current_write_pos)
        
    return result

def get_twiddle(index, W_array, N_total):
    """
    Hardware ROM Simulation:
    Fetches the twiddle factor using only an N/4 sized array.
    Maps quadrant symmetries to complex coordinate swaps.
    """
    # Prevent modulo-by-zero for extremely small test cases (N=2)
    quadrant_size = max(1, N_total // 4)
    
    # Wrap index within a full circle (0 to N-1)
    idx = index % N_total
    
    # Determine which 90-degree sector the index falls into
    quadrant = idx // quadrant_size
    offset = idx % quadrant_size
    
    # Fetch the base 1st-quadrant value from "ROM"
    base_W = W_array[offset]
    R = base_W.real
    I = base_W.imag
    
    # Apply hardware multiplexing/negation based on the quadrant
    if quadrant == 0:
        return base_W               # W
    elif quadrant == 1:
        return complex(I, -R)       # W * (-j) -> Swap buses, negate Real
    elif quadrant == 2:
        return complex(-R, -I)      # W * (-1) -> Negate both buses
    elif quadrant == 3:
        return complex(-I, R)       # W * (+j) -> Swap buses, negate Imaginary

def radix4_butterfly(A, B, C, D):
    """Calculates the Radix-4 butterfly outputs."""
    E = A + C
    F = A - C
    G = B + D
    H = B - D
    
    X0 = E + G
    X2 = E - G
    X1 = F - 1j * H
    X3 = F + 1j * H
    
    return X0, X1, X2, X3

def simulate_optimized_mixed_radix_fft(x_in):
    """Simulates the generic Pipeline utilizing an N/4 sized ROM."""
    N = len(x_in)
    # print(f"Transform Size (N): {N}")
    num_bits = int(math.log2(N))
    if 2**num_bits != N:
        raise ValueError("Transform size N must be a power of 2.")
    
    # ---------------------------------------------------------
    # HARDWARE MEMORY ALLOCATION
    # Only compute and store N/4 values (minimum 1 for safety)
    # ---------------------------------------------------------
    rom_size = max(1, N // 4)
    W = [cmath.exp(-2j * cmath.pi * k / N) for k in range(rom_size)]
    # print(f"Allocated Twiddle ROM Size: {len(W)} values")
    
    # STAGE 0: Input Reordering
    X = [0j] * N
    for i in range(N):
        X[generic_mixed_radix_reorder(i, num_bits)] = x_in[i]
        
    # PIPELINE STAGE GENERATION
    pipeline_stages = []
    L_current = 1
    
    for _ in range(num_bits // 2):
        M = L_current
        L = M * 4
        pipeline_stages.append((4, M, L))
        L_current = L
        
    if num_bits % 2 != 0:
        M = L_current
        L = M * 2
        pipeline_stages.append((2, M, L))

    print("STAGES")
    print(pipeline_stages)

    # EXECUTE PIPELINE
    for stage_idx, (R, M, L) in enumerate(pipeline_stages):
        next_X = X.copy()
        twiddle_step = N // L
        
        print("-" * 50)
        print(f"Stage: {stage_idx + 1} | Radix: {R} | Twiddle Step: {twiddle_step}")

        for k in range(0, N, L):
            for j in range(M):
                if R == 4:
                    idx0, idx1 = k + j, k + j + M
                    idx2, idx3 = k + j + 2*M, k + j + 3*M
                    
                    A = X[idx0]
                    # Fetch twiddles using the N/4 lookup logic
                    B = X[idx1] * get_twiddle(1 * j * twiddle_step, W, N)
                    C = X[idx2] * get_twiddle(2 * j * twiddle_step, W, N)
                    D = X[idx3] * get_twiddle(3 * j * twiddle_step, W, N)

                    print((k//L)*M+j,":",end=" ")
                    print("Indices: ",idx0,idx1,idx2,idx3)
                    
                    X0, X1, X2, X3 = radix4_butterfly(A, B, C, D)
                    
                    next_X[idx0] = X0
                    next_X[idx1] = X1
                    next_X[idx2] = X2
                    next_X[idx3] = X3
                    
                elif R == 2:
                    idx0, idx1 = k + j, k + j + M
                    
                    A = X[idx0]
                    B = X[idx1] * get_twiddle(1 * j * twiddle_step, W, N)

                    print((k//L)*M+j,":",end=" ")
                    print("Indices: ",idx0,idx1)
                    
                    next_X[idx0] = A + B
                    next_X[idx1] = A - B
                    
        X = next_X
        
    return X

# =====================================================================
# Test and Verification
# =====================================================================
if __name__ == "__main__":
    N = 512
    
    np.random.seed(100)
    x = [
        1.0 + math.sin(2 * math.pi * 5 * i / N) + 
        complex(np.random.normal(0, 0.1), np.random.normal(0, 0.1))
        for i in range(N)
    ]
    
    print(f"\n--- N={N} N/4-Optimized Mixed-Radix FFT Simulation ---")
    hw_output = simulate_optimized_mixed_radix_fft(x)
    
    golden_output = np.fft.fft(x)
    differences = np.array(hw_output) - golden_output
    max_error = np.max(np.abs(differences))
    
    print("-" * 50)
    print(f"Max Absolute Error vs Numpy: {max_error:.4e}")
    if max_error < 1e-10:
        print("Status: SUCCESS (Logic perfectly matches Golden Reference)")


--- N=512 N/4-Optimized Mixed-Radix FFT Simulation ---
STAGES
[(4, 1, 4), (4, 4, 16), (4, 16, 64), (4, 64, 256), (2, 256, 512)]
--------------------------------------------------
Stage: 1 | Radix: 4 | Twiddle Step: 128
0 : Indices:  0 1 2 3
1 : Indices:  4 5 6 7
2 : Indices:  8 9 10 11
3 : Indices:  12 13 14 15
4 : Indices:  16 17 18 19
5 : Indices:  20 21 22 23
6 : Indices:  24 25 26 27
7 : Indices:  28 29 30 31
8 : Indices:  32 33 34 35
9 : Indices:  36 37 38 39
10 : Indices:  40 41 42 43
11 : Indices:  44 45 46 47
12 : Indices:  48 49 50 51
13 : Indices:  52 53 54 55
14 : Indices:  56 57 58 59
15 : Indices:  60 61 62 63
16 : Indices:  64 65 66 67
17 : Indices:  68 69 70 71
18 : Indices:  72 73 74 75
19 : Indices:  76 77 78 79
20 : Indices:  80 81 82 83
21 : Indices:  84 85 86 87
22 : Indices:  88 89 90 91
23 : Indices:  92 93 94 95
24 : Indices:  96 97 98 99
25 : Indices:  100 101 102 103
26 : Indices:  104 105 106 107
27 : Indices:  108 109 110 111
28 : Indices:  112 113 114 115
2

ARITHMETIC REDUNDANCY CHECK

In [15]:
import random

# Number of random test cases
test_cases = 10000
all_match = True

for _ in range(test_cases):
    # Generate random integers for a, b, and c (including negative numbers)
    a = random.randint(-100000, 100000)
    b = random.randint(-100000, 100000)
    c = random.randint(-100000, 100000)
    
    # Calculate both sides
    y = a * b + c
    result_1 = y % 3
    result_2 = ((a % 3) * (b % 3) + (c % 3)) % 3
    
    # Check if they mismatch
    if result_1 != result_2:
        all_match = False
        print(f"Mismatch found! a={a}, b={b}, c={c}")
        break

if all_match:
    print(f"✓ Success! Tested {test_cases} random combinations and 100% of them matched.")

✓ Success! Tested 10000 random combinations and 100% of them matched.


PROPOSED R4MDC

In [27]:
import numpy as np

def digit_reverse_radix4(x):
    """
    Rearranges the final 64-point sequence in base-4 digit-reversed order.
    Required for Radix-4 DIF FFTs.
    """
    N = len(x)
    num_digits = int(np.round(np.log(N) / np.log(4)))
    result = np.zeros_like(x, dtype=complex)
    
    for i in range(N):
        # Convert index to base 4 string, reverse it, convert back to integer
        base4_str = np.base_repr(i, base=4).zfill(num_digits)
        rev_i = int(base4_str[::-1], 4)
        result[rev_i] = x[i]
        
    return result

def radix4_butterfly(x0, x1, x2, x3, tw1, tw2, tw3):
    """
    Simulates the hardware of a 4-point Radix-4 DIF butterfly.
    Takes 4 parallel inputs and applies 3 distinct twiddle factors.
    """
    # 4-point DFT Adders/Subtractors
    t0 = x0 + x1 + x2 + x3
    t1 = x0 - 1j*x1 - x2 + 1j*x3
    t2 = x0 - x1 + x2 - x3
    t3 = x0 + 1j*x1 - x2 - 1j*x3
    
    # Complex Multipliers at the lower ports
    return t0, t1 * tw1, t2 * tw2, t3 * tw3

def radix4_fft_sim(x):
    """
    Simulates a 64-point Radix-4 DIF FFT architecture.
    """
    x = np.asarray(x, dtype=complex)
    N = len(x)
    
    # ==========================================
    # STAGE 1 (Data Gap = N/4 = 16)
    # ==========================================
    # In hardware, commutators would separate the stream into 4 delayed paths
    dp0 = x[0:16]
    dp1 = x[16:32]
    dp2 = x[32:48]
    dp3 = x[48:64]
    
    # Generate twiddle factors for Stage 1
    k1 = np.arange(16)
    tw1_1 = np.exp(-2j * np.pi * k1 / N)
    tw1_2 = np.exp(-2j * np.pi * 2 * k1 / N)
    tw1_3 = np.exp(-2j * np.pi * 3 * k1 / N)
    
    # Process the first stage in parallel 
    st1_0, st1_1, st1_2, st1_3 = radix4_butterfly(dp0, dp1, dp2, dp3, tw1_1, tw1_2, tw1_3)
    
    # Pack into memory buffer for the next stage routing
    st1_out = np.zeros_like(x)
    st1_out[0:16]  = st1_0
    st1_out[16:32] = st1_1
    st1_out[32:48] = st1_2
    st1_out[48:64] = st1_3
    
    # ==========================================
    # STAGE 2 (Data Gap = 4)
    # ==========================================
    # The architecture now processes four independent 16-point sequences
    st2_out = np.zeros_like(x)
    k2 = np.arange(4)
    tw2_1 = np.exp(-2j * np.pi * k2 / 16)
    tw2_2 = np.exp(-2j * np.pi * 2 * k2 / 16)
    tw2_3 = np.exp(-2j * np.pi * 3 * k2 / 16)
    
    for i in range(4): 
        offset = i * 16
        sub_dp0 = st1_out[offset : offset+4]
        sub_dp1 = st1_out[offset+4 : offset+8]
        sub_dp2 = st1_out[offset+8 : offset+12]
        sub_dp3 = st1_out[offset+12 : offset+16]
        
        b0, b1, b2, b3 = radix4_butterfly(sub_dp0, sub_dp1, sub_dp2, sub_dp3, tw2_1, tw2_2, tw2_3)
        
        st2_out[offset : offset+4]   = b0
        st2_out[offset+4 : offset+8]   = b1
        st2_out[offset+8 : offset+12]  = b2
        st2_out[offset+12 : offset+16] = b3

    # ==========================================
    # STAGE 3 (Data Gap = 1)
    # ==========================================
    # The architecture processes sixteen independent 4-point sequences
    st3_out = np.zeros_like(x)
    
    # Twiddle factors for N=4 evaluate to 1
    tw3_1 = np.ones(1)
    tw3_2 = np.ones(1)
    tw3_3 = np.ones(1)
    
    for i in range(16):
        offset = i * 4
        sub_dp0 = st2_out[offset : offset+1]
        sub_dp1 = st2_out[offset+1 : offset+2]
        sub_dp2 = st2_out[offset+2 : offset+3]
        sub_dp3 = st2_out[offset+3 : offset+4]
        
        b0, b1, b2, b3 = radix4_butterfly(sub_dp0, sub_dp1, sub_dp2, sub_dp3, tw3_1, tw3_2, tw3_3)
        
        st3_out[offset : offset+1]   = b0
        st3_out[offset+1 : offset+2]   = b1
        st3_out[offset+2 : offset+3]  = b2
        st3_out[offset+3 : offset+4] = b3

    # Return final output routed through digit-reversal
    return digit_reverse_radix4(st3_out)

# ==========================================
# Test Bench
# ==========================================
if __name__ == "__main__":
    # Generate a random 64-point complex test signal
    test_signal = np.random.rand(64) + 1j * np.random.rand(64)
    
    # Run our Radix-4 datapath simulation
    custom_fft_result = radix4_fft_sim(test_signal)
    
    # Run standard NumPy FFT for verification
    numpy_fft_result = np.fft.fft(test_signal)
    
    # Calculate difference
    diff = np.max(np.abs(custom_fft_result - numpy_fft_result))
    
    print(f"Maximum quantization error: {diff:.2e}")
    if diff < 1e-10:
        print("Success! The Radix-4 architecture simulation perfectly matches the theoretical FFT.")
    else:
        print("Error threshold exceeded.")

Maximum quantization error: 3.55e-15
Success! The Radix-4 architecture simulation perfectly matches the theoretical FFT.


128-FFT (4,4,4,2) MDC

In [3]:
import cmath

def w(n, k):
    """Generates the complex twiddle factor W_n^k"""
    angle = -2j * cmath.pi * k / n
    return cmath.exp(angle)

def radix4_butterfly(a, b, c, d, w1, w2, w3):
    """
    Hardware Radix-4 Butterfly Unit (DIF Architecture).
    Additions happen FIRST, Twiddle multiplications happen LAST.
    """
    # Step 1: First stage additions
    s0 = a + c
    s1 = a - c
    s2 = b + d
    s3 = b - d
    
    # Step 2: Second stage additions (with +/- j cross-connections)
    A_pre = s0 + s2
    B_pre = s1 - 1j * s3
    C_pre = s0 - s2
    D_pre = s1 + 1j * s3
    
    # Step 3: Twiddle factor multiplication
    A = A_pre       # w0 is always 1
    B = B_pre * w1
    C = C_pre * w2
    D = D_pre * w3

    # print(A,B,C,D)
    
    return A, B, C, D

def radix2_butterfly(a, b):
    """
    Hardware Radix-2 Finisher Unit.
    Used in Stage 4 where W_2^0 = 1, requiring 0 multipliers.
    """
    return a + b, a - b

def simulate_mdc_pipeline(input_data):
    """
    Simulates the 4-Stage MDC pipeline for a 128-point input.
    The loops represent the data groupings that the delay lines and 
    commutators create to feed the parallel butterfly units.
    """
    assert len(input_data) == 128, "Input sequence must be 128 points."
    
    # ---------------------------------------------------------
    # STAGE 1: Radix-4 (128 -> 32)
    # Hardware Context: Spatial Stride is 32. 
    # Delay lines of 0, 8, 16, 24 cycles align these 4 points.
    # ---------------------------------------------------------
    stage1_out = [0j] * 128
    print("---------------------------STAGE 1----------------------------------------")
    for i in range(32):
        a, b = input_data[i], input_data[i + 32]
        c, d = input_data[i + 64], input_data[i + 96]
        
        w1, w2, w3 = w(128, i), w(128, 2*i), w(128, 3*i)
        # print("Clk: ",i)
        A, B, C, D = radix4_butterfly(a, b, c, d, w1, w2, w3)
        stage1_out[i], stage1_out[i + 32] = A, B
        stage1_out[i + 64], stage1_out[i + 96] = C, D

    # for i in range(128):
        # print(i,": ",stage1_out[i])
    print("---------------------------------------------------------------------------")

    # ---------------------------------------------------------
    # STAGE 2: Radix-4 (32 -> 8)
    # Hardware Context: Spatial Stride is 8.
    # Commutator switches every 2 cycles. Delay lines scale down.
    # ---------------------------------------------------------
    stage2_out = [0j] * 128
    for block in range(4): # 4 independent 32-point blocks
        offset = block * 32
        for i in range(8):
            a, b = stage1_out[offset + i], stage1_out[offset + i + 8]
            c, d = stage1_out[offset + i + 16], stage1_out[offset + i + 24]
            
            w1, w2, w3 = w(32, i), w(32, 2*i), w(32, 3*i)
            
            A, B, C, D = radix4_butterfly(a, b, c, d, w1, w2, w3)
            stage2_out[offset + i], stage2_out[offset + i + 8] = A, B
            stage2_out[offset + i + 16], stage2_out[offset + i + 24] = C, D

    # ---------------------------------------------------------
    # STAGE 3: Radix-4 (8 -> 2)
    # Hardware Context: Spatial Stride is 2.
    # Commutator switches every clock cycle (ping-pong routing).
    # ---------------------------------------------------------
    stage3_out = [0j] * 128
    for block in range(16): # 16 independent 8-point blocks
        offset = block * 8
        for i in range(2):
            a, b = stage2_out[offset + i], stage2_out[offset + i + 2]
            c, d = stage2_out[offset + i + 4], stage2_out[offset + i + 6]
            
            w1, w2, w3 = w(8, i), w(8, 2*i), w(8, 3*i)
            
            A, B, C, D = radix4_butterfly(a, b, c, d, w1, w2, w3)
            stage3_out[offset + i], stage3_out[offset + i + 2] = A, B
            stage3_out[offset + i + 4], stage3_out[offset + i + 6] = C, D

    # ---------------------------------------------------------
    # STAGE 4: Radix-2 Finisher (2 -> 1)
    # Hardware Context: Spatial Stride is 1.
    # Requires 0 cycle delays. Wires feed directly into Radix-2 BFs.
    # ---------------------------------------------------------
    final_output = [0j] * 128
    for block in range(64): # 64 independent 2-point blocks
        offset = block * 2
        a, b = stage3_out[offset], stage3_out[offset + 1]
        
        # Radix-2 finisher requires no twiddle multiplication here
        A, B = radix2_butterfly(a, b)
        final_output[offset], final_output[offset + 1] = A, B

    return final_output

# ==========================================
# Testbench: Verifying the Hardware Model
# ==========================================
if __name__ == "__main__":
    import numpy as np
    
    # 1. Generate a test signal (e.g., a simple sine wave)
    n_points = 128
    t = np.linspace(0, 1, n_points, endpoint=False)
    test_signal = np.sin(2 * np.pi * 10 * t) + 1j * 0 # 10 Hz tone

    print("TEST SIGNAL\n")
    print(test_signal)
    
    # 2. Run the signal through our hardware pipeline simulation
    hardware_fft_result = simulate_mdc_pipeline(list(test_signal))
    
    # 3. Handle Bit-Reversal (Digit-Reversal for Mixed Radix)
    # In a real FPGA, this is handled by writing the sequential output 
    # into a RAM block and reading it out using reversed address lines.
    def digit_reverse_128(index):
        """
        Correct digit reversal for a (4,4,4,2) decimation.
        128 is 7 bits. The physical array bits are laid out as:
        [Stage 1: b6,b5] [Stage 2: b4,b3] [Stage 3: b2,b1] [Stage 4: b0]
        We must reverse the order of these specific chunks.
        """
        stage1 = (index >> 5) & 0x3  # Extracts bits 6 and 5
        stage2 = (index >> 3) & 0x3  # Extracts bits 4 and 3
        stage3 = (index >> 1) & 0x3  # Extracts bits 2 and 1
        stage4 = index & 0x1         # Extracts bit 0
        
        # Reassemble in reverse order: [Stage 4] [Stage 3] [Stage 2] [Stage 1]
        return (stage4 << 6) | (stage3 << 4) | (stage2 << 2) | stage1
    ordered_hardware_result = [0j] * 128
    for i in range(128):
        ordered_hardware_result[digit_reverse_128(i)] = hardware_fft_result[i]
        
    # 4. Compare against standard Numpy FFT to prove correctness
    numpy_fft_result = np.fft.fft(test_signal)
    
    # Calculate max error between our hardware model and float64 math
    error = np.max(np.abs(np.array(ordered_hardware_result) - numpy_fft_result))
    
    print(f"Simulation Complete.")
    print(f"Maximum discrepancy compared to Numpy: {error:.4e}")
    if error < 1e-10:
        print("Hardware Pipeline Model verified successfully!")

TEST SIGNAL

[ 0.00000000e+00+0.j  4.71396737e-01+0.j  8.31469612e-01+0.j
  9.95184727e-01+0.j  9.23879533e-01+0.j  6.34393284e-01+0.j
  1.95090322e-01+0.j -2.90284677e-01+0.j -7.07106781e-01+0.j
 -9.56940336e-01+0.j -9.80785280e-01+0.j -7.73010453e-01+0.j
 -3.82683432e-01+0.j  9.80171403e-02+0.j  5.55570233e-01+0.j
  8.81921264e-01+0.j  1.00000000e+00+0.j  8.81921264e-01+0.j
  5.55570233e-01+0.j  9.80171403e-02+0.j -3.82683432e-01+0.j
 -7.73010453e-01+0.j -9.80785280e-01+0.j -9.56940336e-01+0.j
 -7.07106781e-01+0.j -2.90284677e-01+0.j  1.95090322e-01+0.j
  6.34393284e-01+0.j  9.23879533e-01+0.j  9.95184727e-01+0.j
  8.31469612e-01+0.j  4.71396737e-01+0.j  6.12323400e-16+0.j
 -4.71396737e-01+0.j -8.31469612e-01+0.j -9.95184727e-01+0.j
 -9.23879533e-01+0.j -6.34393284e-01+0.j -1.95090322e-01+0.j
  2.90284677e-01+0.j  7.07106781e-01+0.j  9.56940336e-01+0.j
  9.80785280e-01+0.j  7.73010453e-01+0.j  3.82683432e-01+0.j
 -9.80171403e-02+0.j -5.55570233e-01+0.j -8.81921264e-01+0.j
 -1.0000000

128-FFT (4,4,4,4,2) in Q1.15

In [3]:
import cmath
import numpy as np

# ==========================================
# Fixed-Point Math Library (16-bit Q1.15)
# ==========================================
Q_FRAC = 15
MAX_VAL = (1 << 15) - 1   #  32767
MIN_VAL = -(1 << 15)      # -32768

def float_to_q1_15(val):
    """Converts a float in range [-1.0, 1.0) to a 16-bit Q1.15 integer."""
    scaled = round(val * (1 << Q_FRAC))
    return max(MIN_VAL, min(MAX_VAL, int(scaled))) # Saturation logic

def q1_15_to_float(val):
    """Converts a 16-bit Q1.15 integer back to a standard float."""
    return val / (1 << Q_FRAC)

def complex_to_fixed(c_val):
    """Quantizes a complex float into a complex tuple of Q1.15 integers."""
    return complex(float_to_q1_15(c_val.real), float_to_q1_15(c_val.imag))

def fixed_to_complex(c_val):
    """Recovers a complex float from a tuple of Q1.15 integers."""
    return complex(q1_15_to_float(c_val.real), q1_15_to_float(c_val.imag))

def w_fixed(n, k):
    """Generates the twiddle factor directly in 16-bit Q1.15 format."""
    angle = -2j * cmath.pi * k / n
    c_val = cmath.exp(angle)
    return complex_to_fixed(c_val)

def shift_and_round(val, shift):
    """
    Simulates hardware Arithmetic Shift Right (ASR) with Convergent Rounding.
    Adds a rounding bit (1 << (shift - 1)) before shifting.
    """
    if shift > 0:
        val = (int(val) + (1 << (shift - 1))) >> shift
    return max(MIN_VAL, min(MAX_VAL, int(val)))

def fixed_complex_mult_and_scale(data, twiddle, scale_shift):
    """
    Simulates the Hardware Multiplier and Output Quantizer.
    data: up to 18-bit integer
    twiddle: 16-bit Q1.15 integer
    """
    dr, di = int(data.real), int(data.imag)
    tr, ti = int(twiddle.real), int(twiddle.imag)
    
    # 1. The 18x16 Hardware Multipliers
    real_part = (dr * tr) - (di * ti)
    imag_part = (dr * ti) + (di * tr)
    
    # 2. Total Shift = Format Recovery (15 bits) + Stage Scaling (e.g., 2 bits)
    total_shift = Q_FRAC + scale_shift
    
    # 3. Round and clamp back to 16 bits
    r_out = shift_and_round(real_part, total_shift)
    i_out = shift_and_round(imag_part, total_shift)
    
    return complex(r_out, i_out)

# ==========================================
# Hardware Butterfly Units
# ==========================================
def radix4_butterfly_fixed(a, b, c, d, w1, w2, w3):
    """
    Bit-Accurate Radix-4 Butterfly.
    All inputs and outputs are strictly 16-bit Q1.15.
    Internal datapath grows to 18 bits temporarily.
    """
    # Step 1: Additions (Growth from 16 to 17 bits)
    s0 = a + c
    s1 = a - c
    s2 = b + d
    s3 = b - d
    
    # Step 2: +/- j cross-connections (Growth from 17 to 18 bits)
    A_pre = s0 + s2
    B_pre = s1 - 1j * s3 
    C_pre = s0 - s2
    D_pre = s1 + 1j * s3
    
    # Step 3: Twiddle Multiplication and Stage Scaling (Divide by 4)
    # A_pre requires no twiddle, so we just shift right by 2 to scale.
    A_r = shift_and_round(A_pre.real, 2)
    A_i = shift_and_round(A_pre.imag, 2)
    A = complex(A_r, A_i)
    
    # B, C, D pass through the 18x16 multipliers and are scaled back to 16 bits
    B = fixed_complex_mult_and_scale(B_pre, w1, 2)
    C = fixed_complex_mult_and_scale(C_pre, w2, 2)
    D = fixed_complex_mult_and_scale(D_pre, w3, 2)
    
    return A, B, C, D

def radix2_butterfly_fixed(a, b):
    """
    Bit-Accurate Radix-2 Finisher.
    Internal addition grows to 17 bits, scales down to 16 bits (Divide by 2).
    """
    s0 = a + b
    s1 = a - b
    
    # Shift right by 1 bit to prevent overflow
    A_r = shift_and_round(s0.real, 1)
    A_i = shift_and_round(s0.imag, 1)
    
    B_r = shift_and_round(s1.real, 1)
    B_i = shift_and_round(s1.imag, 1)
    
    return complex(A_r, A_i), complex(B_r, B_i)

# ==========================================
# MDC Pipeline Datapath
# ==========================================
def simulate_mdc_pipeline_fixed(input_data):
    """
    Simulates the fixed-width 128-point pipeline. 
    Memory depth remains strictly 16 bits wide everywhere.
    """
    assert len(input_data) == 128, "Input sequence must be 128 points."
    
    # STAGE 1: Radix-4 (128 -> 32)
    stage1_out = [0j] * 128
    for i in range(32):
        a, b = input_data[i], input_data[i + 32]
        c, d = input_data[i + 64], input_data[i + 96]
        w1, w2, w3 = w_fixed(128, i), w_fixed(128, 2*i), w_fixed(128, 3*i)
        
        A, B, C, D = radix4_butterfly_fixed(a, b, c, d, w1, w2, w3)
        stage1_out[i], stage1_out[i + 32] = A, B
        stage1_out[i + 64], stage1_out[i + 96] = C, D

    # STAGE 2: Radix-4 (32 -> 8)
    stage2_out = [0j] * 128
    for block in range(4): 
        offset = block * 32
        for i in range(8):
            a, b = stage1_out[offset + i], stage1_out[offset + i + 8]
            c, d = stage1_out[offset + i + 16], stage1_out[offset + i + 24]
            w1, w2, w3 = w_fixed(32, i), w_fixed(32, 2*i), w_fixed(32, 3*i)
            
            A, B, C, D = radix4_butterfly_fixed(a, b, c, d, w1, w2, w3)
            stage2_out[offset + i], stage2_out[offset + i + 8] = A, B
            stage2_out[offset + i + 16], stage2_out[offset + i + 24] = C, D

    # STAGE 3: Radix-4 (8 -> 2)
    stage3_out = [0j] * 128
    for block in range(16): 
        offset = block * 8
        for i in range(2):
            a, b = stage2_out[offset + i], stage2_out[offset + i + 2]
            c, d = stage2_out[offset + i + 4], stage2_out[offset + i + 6]
            w1, w2, w3 = w_fixed(8, i), w_fixed(8, 2*i), w_fixed(8, 3*i)
            
            A, B, C, D = radix4_butterfly_fixed(a, b, c, d, w1, w2, w3)
            stage3_out[offset + i], stage3_out[offset + i + 2] = A, B
            stage3_out[offset + i + 4], stage3_out[offset + i + 6] = C, D

    # STAGE 4: Radix-2 Finisher (2 -> 1)
    final_output = [0j] * 128
    for block in range(64): 
        offset = block * 2
        a, b = stage3_out[offset], stage3_out[offset + 1]
        
        A, B = radix2_butterfly_fixed(a, b)
        final_output[offset], final_output[offset + 1] = A, B

    return final_output

# ==========================================
# Testbench
# ==========================================
if __name__ == "__main__":
    
    # 1. Generate standard float signal (Max amplitude = 1.0)
    n_points = 128
    t = np.linspace(0, 1, n_points, endpoint=False)
    test_signal = np.sin(2 * np.pi * 10 * t) + 1j * 0
    
    # 2. Quantize incoming signal for hardware (Simulates the ADC)
    fixed_signal = [complex_to_fixed(val) for val in test_signal]
    
    # 3. Run the Bit-Accurate Pipeline
    hardware_fft_result = simulate_mdc_pipeline_fixed(fixed_signal)
    
    # 4. Handle Bit-Reversal (4,4,4,2)
    def digit_reverse_128(index):
        stage1 = (index >> 5) & 0x3 
        stage2 = (index >> 3) & 0x3 
        stage3 = (index >> 1) & 0x3 
        stage4 = index & 0x1        
        return (stage4 << 6) | (stage3 << 4) | (stage2 << 2) | stage1
        
    ordered_hardware_result = [0j] * 128
    for i in range(128):
        ordered_hardware_result[digit_reverse_128(i)] = hardware_fft_result[i]
        
    # 5. Extract Hardware Data and Restore Original Scale
    # Our hardware scaled the data down by 128 (4*4*4*2) to prevent bit-growth.
    # To compare apples-to-apples with Numpy, we must multiply by 128.
    float_hw_result = [fixed_to_complex(val) * 128.0 for val in ordered_hardware_result]
    
    # 6. Compare with Numpy
    numpy_fft_result = np.fft.fft(test_signal)
    error = np.max(np.abs(np.array(float_hw_result) - numpy_fft_result))
    
    print(f"Hardware Simulation Complete.")
    print(f"Max Absolute Error vs Numpy: {error:.4f}")
    if error < 1.0:
        print("Model Verified! Error is purely due to 16-bit Quantization Noise.")

    error_vector = float_hw_result - numpy_fft_result
    
    # Power = Sum of the squared magnitudes
    signal_power = np.sum(np.abs(numpy_fft_result)**2)
    noise_power = np.sum(np.abs(error_vector)**2)
    
    if noise_power == 0:
        sqnr_db = float('inf')
    else:
        sqnr_db = 10 * np.log10(signal_power / noise_power)
    
    max_error = np.max(np.abs(error_vector))
    
    print("==================================================")
    print("   HARDWARE FFT SIMULATION RESULTS (16-bit Q1.15) ")
    print("==================================================")
    print(f"Max Absolute Error: {max_error:.4f}")
    print(f"Signal Power:       {signal_power:.4f}")
    print(f"Noise Power:        {noise_power:.4f}")
    print(f"Final SQNR:         {sqnr_db:.2f} dB")
    print("==================================================")



Hardware Simulation Complete.
Max Absolute Error vs Numpy: 0.0039
Model Verified! Error is purely due to 16-bit Quantization Noise.
   HARDWARE FFT SIMULATION RESULTS (16-bit Q1.15) 
Max Absolute Error: 0.0039
Signal Power:       8192.0000
Noise Power:        0.0000
Final SQNR:         87.30 dB
